<a href="https://colab.research.google.com/github/TianaT6/MSSP607/blob/main/Modules/Week_10/MSSP607_Week10_Improved.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MSSP607 — Week 10: Improving Coding

**Student:** Tianyi Tan  
**Original notebook:** `MSSP607_Week5_tt (1).ipynb`  
**AI tool:** ChatGPT

This version improves my Week 5 stock-data notebook while keeping tasks
2(b)–2(h), the historical ticker `FB`, and the filename `faanf.csv`.

## AI recommendations

| Original approach | Improvement | Reason |
|---|---|---|
| Repeat loading and ticker statements five times | Use a function and loop | Keep the same steps for every company |
| Assume input files and values are valid | Add file, column, and data checks | Identify problems before analysis |
| Save and reopen the combined data before sorting | Keep the DataFrame in memory | Avoid an unnecessary file read |
| Melt every remaining column | List the five measurement columns | Make the intended transformation clear |
| Include a long encoded data block | Keep CSV files separate from the code | Make the notebook easier to read |

**Request to ChatGPT:** Review my Week 5 Python code, recommend improvements,
and create an improved Jupyter Notebook that keeps the original analysis.

## Before running

1. Unzip `Week10_CSV_Files.zip`. It contains the five original input files:
   `aapl.csv`, `amzn.csv`, `nflx.csv`, `fb.csv`, and `goog.csv`.
2. In **Colab**, upload these five CSVs using the Files sidebar. In **Jupyter**,
   keep them beside this notebook or in a `data` subfolder.
3. Run all cells from top to bottom. Result CSVs are saved in `week10_outputs`.

The inputs are the exact CSV bytes extracted from the original notebook.
This version contains no encoded data block and does not require a GitHub login.

## 1. Set up paths and column names

The dictionary connects each filename to its ticker. The code accepts input
files in a `data` subfolder or in the current working folder.

In [4]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

STOCK_FILES = {
    "aapl.csv": "AAPL",
    "amzn.csv": "AMZN",
    "nflx.csv": "NFLX",
    "fb.csv": "FB",
    "goog.csv": "GOOG",
}
VALUE_COLUMNS = ["open", "high", "low", "close", "volume"]
REQUIRED_COLUMNS = ["date"] + VALUE_COLUMNS

DATA_DIR = Path("data")
if not all((DATA_DIR / name).is_file() for name in STOCK_FILES):
    DATA_DIR = Path(".")

missing_files = [name for name in STOCK_FILES if not (DATA_DIR / name).is_file()]
if missing_files:
    raise FileNotFoundError(
        "Upload the five original CSV files before running. Missing: "
        + ", ".join(missing_files)
    )

OUTPUT_DIR = Path("week10_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Input folder:", DATA_DIR)
print("Output folder:", OUTPUT_DIR)

Input folder: .
Output folder: week10_outputs


## 2(b)–2(c). Read each CSV and add its ticker

This function replaces the five separate loading statements and five separate
ticker statements. It checks required columns, missing or infinite values,
duplicate dates, and nonnegative whole-number trading volumes. Invalid inputs
raise an error; observations are not silently deleted or filled.

In [5]:
def load_stock_csv(file_path, ticker):
    """Read and validate one stock CSV, then add its ticker."""
    if not file_path.is_file():
        raise FileNotFoundError(f"Cannot find {file_path}.")

    df = pd.read_csv(file_path)
    if df.empty:
        raise ValueError(f"{file_path.name} contains no data.")

    missing_columns = set(REQUIRED_COLUMNS) - set(df.columns)
    if missing_columns:
        raise ValueError(
            f"{file_path.name} is missing columns: {sorted(missing_columns)}"
        )

    df = df[REQUIRED_COLUMNS].copy()
    df["date"] = pd.to_datetime(
        df["date"], format="%Y-%m-%d", errors="raise"
    )
    for column in VALUE_COLUMNS:
        df[column] = pd.to_numeric(df[column], errors="raise")

    if df.isna().any().any():
        raise ValueError(f"{file_path.name} contains missing values.")
    if not np.isfinite(df[VALUE_COLUMNS].to_numpy(dtype=float)).all():
        raise ValueError(f"{file_path.name} contains infinite values.")
    if df["date"].duplicated().any():
        raise ValueError(f"{file_path.name} contains duplicate dates.")

    invalid_volume = (df["volume"] < 0) | (df["volume"] % 1 != 0)
    if invalid_volume.any():
        raise ValueError(f"{file_path.name} contains invalid trading volumes.")

    df["volume"] = df["volume"].astype("int64")
    df["ticker"] = ticker
    return df


stock_frames = []
for filename, ticker in STOCK_FILES.items():
    stock_df = load_stock_csv(DATA_DIR / filename, ticker)
    stock_frames.append(stock_df)
    print(f"{ticker}: {len(stock_df):,} rows")

display(stock_frames[0].head())

AAPL: 251 rows
AMZN: 251 rows
NFLX: 251 rows
FB: 251 rows
GOOG: 251 rows


,date,open,high,low,close,volume,ticker
0,2018-01-02,166.9271,169.0264,166.0442,168.9872,25555934,AAPL
1,2018-01-03,169.2521,171.2337,168.6929,168.9578,29517899,AAPL
2,2018-01-04,169.2619,170.1742,168.8106,169.7426,22434597,AAPL
3,2018-01-05,170.1448,172.0381,169.7622,171.6751,23660018,AAPL
4,2018-01-08,171.0375,172.2736,170.6255,171.0375,20567766,AAPL


## 2(d). Combine the DataFrames

`pd.concat()` stacks the observations and creates a new row index. Each input
stock has 251 rows, so the supplied files produce 1,255 rows and 7 columns.

In [6]:
combined = pd.concat(stock_frames, ignore_index=True)
print("Combined shape:", combined.shape)
display(combined["ticker"].value_counts().sort_index())
display(combined.head())

Combined shape: (1255, 7)


,count
ticker,
AAPL,251
AMZN,251
FB,251
GOOG,251
NFLX,251


,date,open,high,low,close,volume,ticker
0,2018-01-02,166.9271,169.0264,166.0442,168.9872,25555934,AAPL
1,2018-01-03,169.2521,171.2337,168.6929,168.9578,29517899,AAPL
2,2018-01-04,169.2619,170.1742,168.8106,169.7426,22434597,AAPL
3,2018-01-05,170.1448,172.0381,169.7622,171.6751,23660018,AAPL
4,2018-01-08,171.0375,172.2736,170.6255,171.0375,20567766,AAPL


## 2(e). Save `faanf.csv`

I use `faanf.csv` consistently, as in the original notebook. `index=False`
prevents an extra index column from being written to the CSV.

In [7]:
combined.to_csv(
    OUTPUT_DIR / "faanf.csv", index=False, date_format="%Y-%m-%d"
)
print("Saved", OUTPUT_DIR / "faanf.csv")

Saved week10_outputs/faanf.csv


## 2(f). Convert types and sort

Dates and volumes were converted in the loading function. Use the existing
DataFrame instead of reading the combined file again, then sort by date and
ticker in ascending order.

In [8]:
stocks = combined.sort_values(
    ["date", "ticker"]
).reset_index(drop=True)
stocks.to_csv(
    OUTPUT_DIR / "faanf_sorted.csv", index=False, date_format="%Y-%m-%d"
)
print(stocks.dtypes)
print("Date range:", stocks["date"].min().date(), "to", stocks["date"].max().date())
display(stocks.head())

date      datetime64[ns]
open             float64
high             float64
low              float64
close            float64
volume             int64
ticker            object
dtype: object
Date range: 2018-01-02 to 2018-12-31


,date,open,high,low,close,volume,ticker
0,2018-01-02,166.9271,169.0264,166.0442,168.9872,25555934,AAPL
1,2018-01-02,1172.0000,1190.0000,1170.5100,1189.0100,2694494,AMZN
2,2018-01-02,177.6800,181.5800,177.5500,181.4200,18151903,FB
3,2018-01-02,1048.3400,1066.9400,1045.2300,1065.0000,1237564,GOOG
4,2018-01-02,196.1000,201.6500,195.4200,201.0700,10966889,NFLX


CSV files do not preserve pandas types. Convert the date column again when
reopening a CSV for date analysis. The historical `FB` ticker is retained.

## 2(g). Find the seven highest trading volumes

These are the seven largest daily volume observations across all five stocks.
For the supplied data, the largest is 169,803,668 for FB on July 26, 2018.

In [9]:
top7 = stocks.nlargest(7, "volume").reset_index(drop=True)
top7.to_csv(
    OUTPUT_DIR / "top7_volume.csv", index=False, date_format="%Y-%m-%d"
)
display(top7)

,date,open,high,low,close,volume,ticker
0,2018-07-26,174.8900,180.1300,173.7500,176.2600,169803668,FB
1,2018-03-20,167.4700,170.2000,161.9500,168.1500,129851768,FB
2,2018-03-26,160.8200,161.1000,149.0200,160.0600,126116634,FB
3,2018-03-21,164.8000,173.4000,163.3000,169.3900,106598834,FB
4,2018-09-21,219.0727,219.6482,215.6097,215.9768,96246748,AAPL
5,2018-12-21,156.1901,157.4845,148.9909,150.0862,95744384,AAPL
6,2018-11-02,207.9295,211.9978,203.8414,205.8755,91328654,AAPL


## 2(h). Reshape the data with `melt()`

Keep date and ticker as identifiers. Explicitly select the five measurements
so the transformation remains clear. The supplied inputs produce 6,275 rows
and 4 columns.

In [10]:
melted = stocks.melt(
    id_vars=["date", "ticker"],
    value_vars=VALUE_COLUMNS,
    var_name="variable",
    value_name="value",
)
melted.to_csv(
    OUTPUT_DIR / "faanf_melted.csv", index=False, date_format="%Y-%m-%d"
)
print("Melted shape:", melted.shape)
display(melted.head())

Melted shape: (6275, 4)


,date,ticker,variable,value
0,2018-01-02,AAPL,open,166.9271
1,2018-01-02,AMZN,open,1172.0000
2,2018-01-02,FB,open,177.6800
3,2018-01-02,GOOG,open,1048.3400
4,2018-01-02,NFLX,open,196.1000


## 3. Review saved results

The melted `value` column contains both prices and volumes, so it uses a
floating-point type. The wide `stocks` DataFrame retains integer volumes.

In [11]:
results = {
    "faanf.csv": combined,
    "faanf_sorted.csv": stocks,
    "top7_volume.csv": top7,
    "faanf_melted.csv": melted,
}
summary = pd.DataFrame([
    {"file": str(OUTPUT_DIR / filename), "rows": len(table),
     "columns": len(table.columns)}
    for filename, table in results.items()
])
display(summary)

,file,rows,columns
0,week10_outputs/faanf.csv,1255,7
1,week10_outputs/faanf_sorted.csv,1255,7
2,week10_outputs/top7_volume.csv,7,7
3,week10_outputs/faanf_melted.csv,6275,4


## 4. Reflection and submission

The main improvements are less repeated code, consistent data checks, and
clearer file handling. The original analysis tasks are preserved. Input checks
help identify formatting problems; they do not verify the accuracy of the
original stock prices.

For Week 10, upload this improved `.ipynb` file and the five input `.csv` files
to Canvas. Complete the separate Week 10 Discussion according to its own
prompt, and save the notebook and data in the MSSP607 GitHub repository.
Check the instructor's and TAs' collaborator access. Creating these files does
not complete the Canvas or GitHub steps.